# Welcome to Pipelines

In [1]:
# Skip this if packages are already in .venv (this env has no pip module).
# datasets 3.6.0 and transformers are already installed from Day 1.
# %pip install -q --upgrade datasets==3.6.0 transformers==4.57.6
print("Using existing venv packages (datasets, transformers, diffusers)")
 

/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
# create loop to use apple's mps for faster training

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"Device type: {device.type}")


MPS available: True
Device type: mps


In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")

if hf_token and hf_token.startswith("hf_"):
    print("HF key looks good so far")
else:
    raise ValueError("HF_TOKEN not found. Add HF_TOKEN=hf_... to your .env file.")

login(hf_token, add_to_git_credential=True)


HF key looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Using Pipelines from Hugging Face

In [9]:
# Sentiment Analysis

my_simple_sentiment_analyzer = pipeline("sentiment-analysis",
                                        device = device)
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps


[{'label': 'POSITIVE', 'score': 0.9993460774421692}]


In [10]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the wat to LLM mastery!")
print(result)

[{'label': 'POSITIVE', 'score': 0.8994835615158081}]


In [11]:
better_sentiment = pipeline("sentiment-analysis", 
                            model = "nlptown/bert-base-multilingual-uncased-sentiment",
                            device = device)
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use mps


[{'label': '3 stars', 'score': 0.39448052644729614}]


In [12]:
# named entity recognition
ner = pipeline("ner", device = device)
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
    print(entity)


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use mps


{'entity': 'I-ORG', 'score': np.float32(0.999476), 'index': 1, 'word': 'AI', 'start': 0, 'end': 2}
{'entity': 'I-ORG', 'score': np.float32(0.9962089), 'index': 2, 'word': 'Engineers', 'start': 3, 'end': 12}
{'entity': 'I-ORG', 'score': np.float32(0.8205675), 'index': 11, 'word': 'Hu', 'start': 59, 'end': 61}
{'entity': 'I-ORG', 'score': np.float32(0.6514929), 'index': 12, 'word': '##gging', 'start': 61, 'end': 66}
{'entity': 'I-ORG', 'score': np.float32(0.9609928), 'index': 13, 'word': '##F', 'start': 66, 'end': 67}
{'entity': 'I-ORG', 'score': np.float32(0.925171), 'index': 14, 'word': '##ace', 'start': 67, 'end': 70}
{'entity': 'I-MISC', 'score': np.float32(0.8882751), 'index': 16, 'word': 'Google', 'start': 74, 'end': 80}
{'entity': 'I-MISC', 'score': np.float32(0.6730775), 'index': 17, 'word': 'Cola', 'start': 81, 'end': 85}
{'entity': 'I-PER', 'score': np.float32(0.9989441), 'index': 20, 'word': 'Ed', 'start': 92, 'end': 94}
{'entity': 'I-PER', 'score': np.float32(0.99872154), 'in

In [13]:
# Question Answering with Context

question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for interface of LLMs with common tasks"

question_answerer = pipeline("question-answering",
                            device = device)
result = question_answerer(question=question,
                          context=context)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use mps


In [14]:
# Text Summarization

summarizer = pipeline("summarization",
                      device = device)
text = """ 
The Hugging Face transformers library is an incredibly versatile
and powerful tool for natural language processing(NLP). It allows
users to perform a wide range of tasks such as text classification,
named entity recognition, and question answering, among others. It's
an extremely popular library that's widely used by the open-source
data science community. It lowers the barrier to entry into the 
field by providing Data Scientists with a productive, convenient
way to work with transformer models.
"""

summary = summarizer(text,
                                    max_length = 50,
                                    min_length = 25,
                                    do_sample = False)
print(summary[0]["summary_text"])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use mps


 The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing(NLP) It allows users to perform a wide range of tasks such as text classification, categorized entity recognition, and question answering . It's


In [16]:
# Translation

translator = pipeline("translation_en_to_es",
                      model = "Helsinki-NLP/opus-mt-en-es",
                      device = device)
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]["translation_text"])

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps


Los científicos de datos estaban verdaderamente sorprendidos por el poder y la simplicidad de la API de tuberías HuggingFace.
